In [374]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor  # For continuous 0-1 output
from sklearn.model_selection import train_test_split

In [180]:
midip = pd.read_csv('MIDIP_COW.csv')

In [215]:
midip['start_date'] = pd.to_datetime(
    midip[['styear', 'stmon', 'stday']].rename(columns={
        'styear': 'year', 
        'stmon': 'month', 
        'stday': 'day'
    }),
    errors='coerce'
)

midip['end_date'] = pd.to_datetime(
    midip[['endyear', 'endmon', 'endday']].rename(columns={
        'endyear': 'year', 
        'endmon': 'month', 
        'endday': 'day'
    }),
    errors='coerce'
).fillna(midip['start_date'])

# Calculate duration in days (minimum 1 day)
midip['duration'] = (midip['end_date'] - midip['start_date']).dt.days.clip(lower=1)

fatality: Fatality Level — Ordinal scale of deaths caused by the state in the incident:
0 = None,
1 = 1–10 deaths,
2 = 11–100 deaths,
3 = 101–1,000 deaths,
4 = 1,001–10,000 deaths,
5 = More than 10,000 deaths. 

action: Action Code — Type of militarized action taken by the state:
1 = Threat to use force,
2 = Threat to blockade,
3 = Threat to occupy territory,
4 = Show of force,
5 = Use of force,
6 = Blockade,
7 = Occupation of territory.

hostlev: Hostility Level — Highest hostility level reached by the state in the incident:
1 = No militarized action,
2 = Threat to use force,
3 = Display of force,
4 = Use of force,
5 = War.

revtype1: Primary Revision Type — Main issue the state sought to revise:
1 = Territory,
2 = Policy,
3 = Regime,
4 = Other.

In [216]:
indian_disputes= midip[midip['ccode']==750]

In [307]:
def find_disputes(country_code):
    country_disputes = midip[midip['ccode']==country_code]

    common_dispnums = set(indian_disputes['dispnum']).intersection(set(country_disputes['dispnum']))
    number_of_common_disputes = len(common_dispnums)
    common_disputes = sorted(common_dispnums)

    disputes_with_india = indian_disputes[indian_disputes['dispnum'].isin(common_disputes)]
    
    last_conflict_year = disputes_with_india['styear'].max() if not disputes_with_india.empty else 0
    return disputes_with_india, number_of_common_disputes, last_conflict_year

In [218]:
find_disputes(770)[0]

,dispnum,incidnum,stabb,ccode,stday,stmon,styear,endday,endmon,endyear,...,fatality,fatalpre,action,hostlev,revtype1,revtype2,version,start_date,end_date,duration
400,4007,4007001,IND,750,17,9,1993,17,9,1993,...,0,0,17,4,1,0,5,1993-09-17,1993-09-17,1.0
402,4007,4007002,IND,750,7,11,1993,7,11,1993,...,0,0,17,4,1,0,5,1993-11-07,1993-11-07,1.0
404,4007,4007003,IND,750,10,1,1994,12,1,1994,...,0,0,17,4,1,0,5,1994-01-10,1994-01-12,2.0
406,4007,4007004,IND,750,12,5,1994,12,5,1994,...,0,0,0,1,0,0,5,1994-05-12,1994-05-12,1.0
408,4007,4007005,IND,750,29,9,1994,29,9,1994,...,0,0,16,4,1,0,5,1994-09-29,1994-09-29,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7887,4585,4585143,IND,750,22,11,2014,22,11,2014,...,0,0,16,4,1,0,5,2014-11-22,2014-11-22,1.0
7889,4585,4585144,IND,750,27,11,2014,28,11,2014,...,0,0,17,4,1,0,5,2014-11-27,2014-11-28,1.0
7891,4585,4585145,IND,750,23,12,2014,24,12,2014,...,0,0,17,4,1,0,5,2014-12-23,2014-12-24,1.0
7893,4585,4585146,IND,750,27,12,2014,27,12,2014,...,0,0,17,4,1,0,5,2014-12-27,2014-12-27,1.0


In [336]:
def calculate_bilateral_severity(
    country_code,
    decay_rate=0.1,
    current_year=2025,
    random_state=None
):
    try:
        # Get disputes data
        disputes_df, _, _ = find_disputes(country_code)
        if disputes_df.empty or len(disputes_df) < 2:
            return 0.0

        # Validate features
        features = ['hostlev', 'fatality', 'action', 'duration', 'revtype1']
        required_cols = features + ['styear']
        
        missing_cols = [col for col in required_cols if col not in disputes_df.columns]
        if missing_cols:
            raise ValueError(f"Missing columns: {missing_cols}")

        # Preprocess data
        df = disputes_df[required_cols].copy()
        df[features] = df[features].apply(pd.to_numeric, errors='coerce')
        df = df.dropna()
        
        if len(df) < 2:
            return 0.0

        # Normalization and PCA
        scaler = MinMaxScaler()
        scaled_features = scaler.fit_transform(df[features])
        
        if np.isnan(scaled_features).any() or (scaled_features.var(axis=0) == 0).all():
            return 0.0

        pca = PCA(n_components=1, random_state=random_state)
        df['pca_severity'] = pca.fit_transform(scaled_features)

        # Time weighting
        df['years_ago'] = current_year - df['styear']
        df['time_weight'] = np.exp(-decay_rate * df['years_ago'])
        return -1 * (df['pca_severity'] * df['time_weight']).sum()

    except Exception as e:
        print(f"Error processing {country_code}: {str(e)}")
        return 0.0

In [337]:
calculate_bilateral_severity(710)



0.14420217948342673

In [338]:
calculate_bilateral_severity(2)

0.0

In [339]:
calculate_bilateral_severity(770)

2.4693463974880796

In [340]:
def MIDIP_final_data(country_code):
    df ,militarized_disputes , last_conflict_year= find_disputes(country_code)
    conflict_severity_score = calculate_bilateral_severity(country_code)

    return militarized_disputes, last_conflict_year, conflict_severity_score

In [341]:
MIDIP_final_data(770)

(6, 2014, 2.4693463974880796)

In [342]:
MIDIP_final_data(710)

(5, 2014, 0.14420217948342673)



### Now lets move on to Alliance data (1816 - 2012)


In [343]:
alliance_data=pd.read_csv('alliance_COW.csv')

In [344]:
alliance_data

,version4id,ccode1,state_name1,ccode2,state_name2,dyad_st_day,dyad_st_month,dyad_st_year,dyad_end_day,dyad_end_month,dyad_end_year,left_censor,right_censor,defense,neutrality,nonaggression,entente,year,version
0,1,200,United Kingdom,235,Portugal,1,1,1816,NaN,NaN,2012,1,1,1,0,1.0,0.0,1816,4.1
1,1,200,United Kingdom,235,Portugal,1,1,1816,NaN,NaN,2012,1,1,1,0,1.0,0.0,1817,4.1
2,1,200,United Kingdom,235,Portugal,1,1,1816,NaN,NaN,2012,1,1,1,0,1.0,0.0,1818,4.1
3,1,200,United Kingdom,235,Portugal,1,1,1816,NaN,NaN,2012,1,1,1,0,1.0,0.0,1819,4.1
4,1,200,United Kingdom,235,Portugal,1,1,1816,NaN,NaN,2012,1,1,1,0,1.0,0.0,1820,4.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74124,414,2,United States of America,666,Israel,30,11,1987,26.0,12.0,1991,0,0,0,0,0.0,1.0,0,4.1
74125,414,2,United States of America,666,Israel,30,11,1988,26.0,12.0,1991,0,0,0,0,0.0,1.0,0,4.1
74126,414,2,United States of America,666,Israel,30,11,1989,26.0,12.0,1991,0,0,0,0,0.0,1.0,0,4.1
74127,414,2,United States of America,666,Israel,30,11,1990,26.0,12.0,1991,0,0,0,0,0.0,1.0,0,4.1


In [345]:
indian_alliances = alliance_data[
    (alliance_data['state_name1'] == 'India') |
    (alliance_data['state_name1'] == 'India')
].copy()

In [346]:
indian_alliances

,version4id,ccode1,state_name1,ccode2,state_name2,dyad_st_day,dyad_st_month,dyad_st_year,dyad_end_day,dyad_end_month,dyad_end_year,left_censor,right_censor,defense,neutrality,nonaggression,entente,year,version
62961,267,750,India,775,Myanmar,23,7,1962,12.0,7.0,1964,0,0,0,1,0.0,0.0,1962,4.1
62962,267,750,India,775,Myanmar,23,7,1962,12.0,7.0,1964,0,0,0,1,0.0,0.0,1963,4.1
62963,267,750,India,775,Myanmar,23,7,1962,12.0,7.0,1964,0,0,0,1,0.0,0.0,1964,4.1
62964,267,750,India,800,Thailand,23,7,1962,12.0,7.0,1964,0,0,0,1,0.0,0.0,1962,4.1
62965,267,750,India,800,Thailand,23,7,1962,12.0,7.0,1964,0,0,0,1,0.0,0.0,1963,4.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63653,294,750,India,770,Pakistan,2,7,1972,NaN,NaN,2012,0,1,0,0,1.0,0.0,2008,4.1
63654,294,750,India,770,Pakistan,2,7,1972,NaN,NaN,2012,0,1,0,0,1.0,0.0,2009,4.1
63655,294,750,India,770,Pakistan,2,7,1972,NaN,NaN,2012,0,1,0,0,1.0,0.0,2010,4.1
63656,294,750,India,770,Pakistan,2,7,1972,NaN,NaN,2012,0,1,0,0,1.0,0.0,2011,4.1


In [347]:
def check_formal_alliance(country_code):
    alliance_with_country = indian_alliances[indian_alliances["ccode2"] == country_code]
    return 1 if not alliance_with_country.empty else 0


In [348]:
check_formal_alliance(770)


1

In [349]:
check_formal_alliance(710)

0





### Now lets move on to Defence Cooperation data (1980 - 2010)

In [350]:
defence_cooperation = pd.read_csv('defence_cooperation_COW.csv')

In [351]:
indian_defence_cooperation = defence_cooperation[
    (defence_cooperation['ccode1'] == 750) |
    (defence_cooperation['ccode2'] == 750)
].copy()

In [352]:
indian_defence_cooperation

,ccode1,abbrev1,ccode2,abbrev2,year,dcaGeneralV1,dcaGeneralV2,dcaSectorV1,dcaSectorV2,dcaAnyV1,dcaAnyV2
4667,2,USA,750,IND,1980,0,0,0,0,0,0
4668,2,USA,750,IND,1981,0,0,0,0,0,0
4669,2,USA,750,IND,1982,0,0,0,0,0,0
4670,2,USA,750,IND,1983,0,0,0,0,0,0
4671,2,USA,750,IND,1984,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
993703,990,WSM,750,IND,2006,0,0,0,0,0,0
993704,990,WSM,750,IND,2007,0,0,0,0,0,0
993705,990,WSM,750,IND,2008,0,0,0,0,0,0
993706,990,WSM,750,IND,2009,0,0,0,0,0,0


In [353]:
def check_defence_coop(country_code):
    defence_coop_with_ind = indian_defence_cooperation[indian_defence_cooperation['ccode2'] == country_code]
    return 1 if (defence_coop_with_ind['dcaAnyV2'] == 1).any() else 0



In [354]:
check_defence_coop(770)

0

In [355]:
check_defence_coop(2)

1

In [356]:
check_defence_coop(710)

1

### Now lets move on to war data (1816‐2014)


In [357]:
wars_data = pd.read_csv('INTRA_STATE_WARS.csv', encoding='latin1')

In [358]:
wars_data

,WarNum,WarName,V5RegionNum,WarType,CcodeA,SideA,SideB,Intnl,StartMo1,StartDy1,...,WDuratMo,TotNatMonWar,TransFrom,Initiator,Outcome,TransTo,DeathsSideA,DeathsSideB,TotalBDeaths,Version
0,500.0,First Caucasus War of 1818-1822,3,5,365,Russia,Caucasus Rebels,0,6,10,...,53.20,53.20,-8,Chechnya,1,-8.0,5000,6000,11000,5.1
1,502.0,First Two Sicilies War of 1820-1821,3,4,329,Two Sicilies,Liberals,1,7,2,...,8.73,9.03,-8,Liberals,1,-8.0,-9,-9,2000,5.1
2,502.1,Ali Pasha Rebellion of 1820-1822,3,5,640,Ottoman Empire,Ali Pasha Loyalists,0,7,-9,...,18.33,18.33,-8,Ottoman Empire,1,-8.0,-9,-9,2000,5.1
3,503.0,Sardinian Revolt of 1821,3,4,325,Sardinia,Piedmont Liberals,1,3,10,...,1.97,3.03,-8,Liberals,1,-8.0,-9,-9,1000,5.1
4,504.0,Greek Independence War of 1821-1828,3,5,640,Ottoman Empire,Greeks,1,3,25,...,85.03,91.77,-8,Greeks,4,4.0,50000,15177,65177,5.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415,992.0,Third Al-Houthi Rebellion of 2014-present,5,4,679,Yemen,Ansar Allah,0,2,28,...,12.83,12.83,-8,Houthi Rebels,5,-8.0,600,4000,4600,5.1
416,992.5,Somali-Al-Shabaab war of 2014-present,4,4,520,Somalia,Al-Shabaab,1,3,3,...,21.97,131.80,-8,Somalia,5,-8.0,-9,-9,-9,5.1
417,993.0,Donbas War of 2014-present,3,5,369,Ukraine and Russia,Separatists,1,4,15,...,16.43,28.77,-8,Separatists,5,-8.0,3245,2581,5826,5.1
418,994.0,Second Libyan Civil War of 2014-present,5,4,620,Libya,Libyan Dawn,0,5,16,...,19.00,19.07,-8,Libya,5,-8.0,-9,-9,1500,5.1


In [359]:
indian_war_data = wars_data[
    (wars_data['SideA'] == 'India') |
    (wars_data['SideB'] == 'India')
].copy()

## Creating the master table


In [360]:
country_codes = pd.read_csv('COW-country-codes.csv')

In [361]:
country_codes_without_ind = country_codes[country_codes['CCode'] != 750]


In [367]:
def create_master_table(df):
    df=df.copy()

    militarized_disputes_list = []
    last_conflict_year_list = []
    conflict_severity_score_list = []
    formal_alliance_list=[]
    defence_cooperation_list=[]

    for indx, row in df.iterrows():
        country_code = row['CCode']
        militarized_disputes, last_conflict_year, conflict_severity_score = MIDIP_final_data(country_code)
        formal_alliance = check_alliance(country_code)
        defence_cooperation = check_defence_coop(country_code)

        militarized_disputes_list.append(militarized_disputes)
        last_conflict_year_list.append(last_conflict_year)
        conflict_severity_score_list.append(conflict_severity_score)
        formal_alliance_list.append(formal_alliance)
        defence_cooperation_list.append(defence_cooperation)

    df['militarized_disputes'] = militarized_disputes_list
    df['last_conflict_year'] = last_conflict_year_list
    df['conflict_severity_score'] = conflict_severity_score_list
    df['formal_alliance'] = formal_alliance_list
    df['defence_cooperation'] = defence_cooperation_list

    return df

In [368]:
df = create_master_table(country_codes_without_ind)

In [372]:
master_df = df.copy()

In [376]:
def predict_and_rank_conflict_risk(master_df):
    """
    Adds predicted conflict risk (0-1 scale) to the DataFrame and returns:
    - Updated DataFrame with 'predicted_risk' column
    - Feature importance percentages
    """
    # Create working copy to avoid modifying original
    df = master_df.copy()
    
    # 1. Prepare target variable
    df['risk_of_conflict'] = df['conflict_severity_score'].clip(0, 1)
    
    # 2. Define features
    features = [
        'militarized_disputes',
        'last_conflict_year', 
        'formal_alliance',
        'defence_cooperation'
    ]
    
    # 3. Split data (maintain same random state for reproducibility)
    X = df[features]
    y = df['risk_of_conflict']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 4. Train model
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # 5. Predict for all countries
    df['predicted_risk'] = model.predict(X)
    
    # 6. Calculate feature importance
    importance = model.feature_importances_
    feature_weights = pd.Series(importance, index=features)
    feature_weights = (feature_weights / feature_weights.sum() * 100).round(1)
    
    # 7. Sort by risk
    df_sorted = df.sort_values('predicted_risk', ascending=False)
    
    return df_sorted, feature_weights.sort_values(ascending=False)

In [377]:
final_df , feature_weights = predict_and_rank_conflict_risk(master_df)

In [378]:
feature_weights

last_conflict_year      56.1
militarized_disputes    39.1
defence_cooperation      2.6
formal_alliance          2.2
dtype: float64

In [381]:
final_df[final_df['predicted_risk']!=0]

,StateAbb,CCode,StateNme,militarized_disputes,last_conflict_year,conflict_severity_score,formal_alliance,defence_cooperation,risk_of_conflict,predicted_risk
212,PAK,770,Pakistan,6,2014,2.469346e+00,1,0,1.000000,0.650282
202,CHN,710,China,5,2014,1.442022e-01,0,1,0.144202,0.325079
213,BNG,771,Bangladesh,12,2013,-1.477236e-01,1,0,0.000000,0.210094
177,IRN,630,Iran,1,2013,0.000000e+00,0,1,0.000000,0.001442
77,ITA,325,Italy,1,2012,-8.326673e-17,0,1,0.000000,0.001442


In [382]:
final_df

,StateAbb,CCode,StateNme,militarized_disputes,last_conflict_year,conflict_severity_score,formal_alliance,defence_cooperation,risk_of_conflict,predicted_risk
212,PAK,770,Pakistan,6,2014,2.469346e+00,1,0,1.000000,0.650282
202,CHN,710,China,5,2014,1.442022e-01,0,1,0.144202,0.325079
213,BNG,771,Bangladesh,12,2013,-1.477236e-01,1,0,0.000000,0.210094
177,IRN,630,Iran,1,2013,0.000000e+00,0,1,0.000000,0.001442
77,ITA,325,Italy,1,2012,-8.326673e-17,0,1,0.000000,0.001442
...,...,...,...,...,...,...,...,...,...,...
86,ALB,339,Albania,0,0,0.000000e+00,0,0,0.000000,0.000000
87,MNG,341,Montenegro,0,0,0.000000e+00,0,0,0.000000,0.000000
88,MAC,343,Macedonia,0,0,0.000000e+00,0,0,0.000000,0.000000
89,CRO,344,Croatia,0,0,0.000000e+00,0,0,0.000000,0.000000
